# Sequence Labeling: From HMMs to Linear-Chain CRFs

The narrative companion to formalNLP's twenty-first topic. Every number here comes from
`sequence_labeling_hmm_crf.py`, whose harness asserts each claim the page makes. We work on three
testbeds: small chains where we can enumerate every path, two simulated worlds where the Bayes tagger
is exact, and BIO tags over the finance corpus's gold entity spans.

In [ ]:
import sys
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path("notebooks/sequence-labeling-hmm-crf").resolve()))
import sequence_labeling_hmm_crf as sl

np.set_printoptions(precision=4, suppress=True)

## 1. One recursion, two semirings (Thm 1)

A chain scores a path $z_{1:T}$ by $\sum_t U_t(z_t) + \sum_t P_t(z_t, z_{t+1})$. In the $(+, \times)$
semiring the forward pass sums over prefixes and returns $\log Z$; in $(\max, \times)$ it keeps the best
prefix and returns the mode. We check both against brute force on a random chain.

In [ ]:
rng = sl._rng(9000)
U, P = sl._random_chain(rng, 5, 3)
paths, scores = sl.brute_force(U, P)
logZ, node, pair = sl.chain_marginals(U[None], P)
print("log Z  forward:", logZ[0], " brute force:", np.log(np.exp(scores).sum()))
path, best = sl.viterbi(U[None], P)
print("mode   viterbi:", path[0], best[0], " brute force:", paths[scores.argmax()], scores.max())

The HMM case is formalStatistics' forward–backward. On Topic 12's HMM our node marginals equal
the posterior from Topic 12's scaled forward–backward (the harness asserts agreement to $10^{-10}$).

In [ ]:
sl.test_thm1_hmm_case_is_topic12_forward_backward()
print("Topic 12 forward-backward pinned")

## 2. The HMM tagger is counting (Prop 1)

Supervised maximum likelihood normalizes counts; add-$\alpha$ is the posterior mean under a symmetric
Dirichlet($\alpha$) prior. With an end state, the fitted model is a distribution over sequences of every
length: the length law $\nu A^{T-1}e$ sums to one.

In [ ]:
sl.test_prop1_hmm_mle_is_counts()
print("Prop 1 checks pass")

## 3. The CRF's likelihood is concave (Thm 2)

The gradient of the conditional log-likelihood is observed minus expected features, and its Hessian is
$-\sum_i \mathrm{Cov}_\theta[F \mid x_i]$. We get exact Hessian-vector products by the complex step,
$\mathrm{Im}\,\nabla f(\theta + i h v)/h$, through a logsumexp that shifts by the real part's max.

In [ ]:
sl.test_thm2_gradient_hessian_and_concavity()
sl.test_thm2_unique_optimum()
print("Thm 2 checks pass")

## 4. HMMs and indicator CRFs give the same conditionals (Thm 3)

Every HMM's conditional is a CRF with the HMM's log-parameters as weights. Conversely, the h-transform
turns any indicator-feature CRF with start and end weights into an HMM with the same $p(z \mid x)$:
normalize each state's emissions, solve $(cI - M)h = e^{\text{end}}$ for any $c > \rho(M)$, and set
$a_{jk} = M_{jk} h_k / (c\, h_j)$.

In [ ]:
rng = sl._rng(9001)
K, V = 3, 4
trans, start, end, mu = rng.normal(size=(K, K)), rng.normal(size=K), rng.normal(size=K), rng.normal(size=(K, V))
hmm = sl.hmm_from_crf(trans, start, end, mu)
print("rows of A plus stop:", np.exp(hmm.trans).sum(1) + np.exp(hmm.end))
x = rng.integers(0, V, 5)
U = mu[:, x].T.copy(); U[0] += start; U[-1] += end
_, s_crf = sl.brute_force(U, trans)
Uh, Ph = sl.hmm_potentials(hmm, x)
_, s_hmm = sl.brute_force(Uh[0], Ph)
p = lambda s: np.exp(s - np.log(np.exp(s).sum()))
print("max |p_CRF(z|x) - p_HMM(z|x)| over 3^5 paths:", np.abs(p(s_crf) - p(s_hmm)).max())

## 5. Local normalization ignores the future (Prop 2, Cor 2)

An MEMM whose factor at $t$ reads $x_{1:t+w}$ has prefix marginals that no later word can move. So
decoded by posterior marginals it cannot beat the Bayes rule that reads only $x_{1:t+w}$ (Cor 2). Decoded
by Viterbi it can: the sum over futures is one, but their max is not constant.

In [ ]:
for w in sl.WORLDS:
    t = sl.memm_table(w)
    for k, la in enumerate(t["w"]):
        print(f"{w:5} lookahead {la}: ceiling {t['ceiling'][k]:.4f}  MEMM-MBR {t['mbr'][k]:.4f}  "
              f"MEMM-Viterbi {t['viterbi'][k]:.4f}")

## 6. Decoding risk (Thm 4) and the trellis

Viterbi maximizes the probability of the whole path; posterior-marginal decoding maximizes the expected
number of correct tags. The lab's trellis is the first test sequence of the well-specified world where
the two paths differ in at least two positions.

In [ ]:
ex = sl.trellis_example()
print("viterbi:", ex["viterbi"])
print("mbr:    ", ex["mbr"])
print("gold:   ", ex["z"])
for w in sl.WORLDS:
    c = sl.world_curves(w)
    print(f"{w}: Bayes accuracy, MBR {c['bayes_mbr']:.4f} vs Viterbi {c['bayes']:.4f}")

## 7. The structured perceptron (Thm 5)

On data separable with margin $\gamma$, the perceptron makes at most $R^2/\gamma^2$ mistakes. The bound
holds at every margin, and it is loose by orders of magnitude.

In [ ]:
for r in sl.perceptron_sweep():
    print(f"γ ≥ {r['threshold']:.2f}: γ = {r['gamma']:.4f}, R = {r['R']:.3f}, "
          f"mistakes {r['mistakes']:3d}, bound {r['bound']:9.1f}")

## 8. Generative versus discriminative, where the Bayes tagger is exact

In the well-specified world the HMM leads at small $n$ and the indicator CRF ties it at $n = 1000$. In
the next-state world the window CRF leads at every $n$, and the indicator CRF beats the HMM on
conditional log-loss (what it optimizes) while trailing it on accuracy.

In [ ]:
for w in sl.WORLDS:
    c = sl.world_curves(w)
    print(f"\n{w}  (Bayes MBR {c['bayes_mbr']:.4f}, log-loss {c['bayes_nll']:.4f})")
    print("n          ", c["n"])
    for name in ("hmm", "crf0", "crf1", "perceptron"):
        print(f"{name:11}", [round(v, 4) for v in c[name]])
    for name in ("hmm", "crf0", "crf1"):
        print(f"{name + ' nll':11}", [round(v, 4) for v in c[name + "_nll"]])

## 9. Finance: BIO tags over gold entity spans

Seventeen tags (O plus B/I for eight types). MONEY and PERCENT are read off one word after tokenization
(`million`, `percent`), so the work is in names never seen in training.

In [ ]:
for n in sl.FIN_N:
    r = sl.fin_run(n)
    print(f"\n{n} training documents (unseen-word share {r['hmm']['oov_share']:.3f})")
    for name, v in r.items():
        print(f"  {name:12} acc {v['acc']:.4f}  span F1 {v['f1']:.4f}  'the' {v['the']:.4f}  unseen {v['oov']:.4f}")

Prop 2 on a finance sentence. We complete one test sentence's opening with five in-vocabulary
continuations and read the tag of "the". The causal MEMM's marginal never moves; the CRF's does.

In [ ]:
ex = sl.prop2_example()
t = len(sl.PROP2_PREFIX.split()) - 1
for cont, models in ex.items():
    row = [f"{cont:16}"]
    for name, v in models.items():
        k = int(v["node"][t].argmax())
        row.append(f"{name}: {sl.TAGS[k]} {v['node'][t, k]:.3f}")
    print("  ".join(row))

## 10. The harness

Every assertion the page relies on, in one run.

In [ ]:
sl._run_harness()